In [ ]:
# import toml
# from pathlib import Path
# config_path = Path("../../../../configuration/asim_configuration")
# input_config = toml.load(config_path/ "input_configuration.toml")
# summary_config = toml.load(config_path/ "summary_configuration.toml")

For each intermediate stop on a tour (i.e. trip other than the last trip outbound or inbound) each trip is assigned a purpose based on an observed frequency distribution.
The distribution is segmented by tour purpose, tour direction and person type. Work tours are also segmented by departure or arrival time period.

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import os, sys

module_path = os.path.abspath(os.path.join('../../../..')) # or the path to your source code
sys.path.insert(0, module_path)
import util

In [3]:
# read data
person = util.get_person_data(summary_config, uncloned=True)
tour = util.get_tour_data(summary_config)
trip = util.get_trip_data(summary_config)

In [4]:
trip_data = trip.\
    merge(person, how='left',on=['person_id','household_id','source']).\
    merge(tour, how='left',on=['person_id','household_id','tour_id','source']) 

In [5]:
df_plot = trip_data.groupby(['source','purpose'])['trip_weight'].sum().reset_index()
df_plot['percentage'] = df_plot.groupby(['source'], group_keys=False)['trip_weight']. \
    apply(lambda x: x / float(x.sum()))
# df_plot
fig = px.bar(df_plot, x="purpose", y="percentage", color="source",barmode="group",
             title="Trip purpose")
fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
fig.update_layout(height=400, width=700, yaxis=dict(tickformat=".1%"))
fig.show()

C:\Users\Modeller\AppData\Local\Temp\ipykernel_40580\2913760989.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



## Trip purpose by segment

In [6]:
def plot_purpose(df: pd.DataFrame, grp_var: str, grp_var_name: str, n_nol: int, height: int):
    df_plot = df.groupby(['source',grp_var,'purpose'])['trip_weight'].sum().reset_index()
    df_plot['percentage'] = df_plot.groupby(['source',grp_var], group_keys=False)['trip_weight']. \
        apply(lambda x: x / float(x.sum()))

    fig = px.bar(df_plot,
                 x="percentage", y="purpose", color="source",barmode="group",
                 facet_col=grp_var, facet_col_wrap=n_nol, orientation='h',
                 title="Trip purpose by " + grp_var_name)
    fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
    fig.update_layout(height=height, width=770)
    fig.for_each_xaxis(lambda a: a.update(tickformat = ".1%"))
    fig.update_yaxes()
    fig.show()

In [7]:
plot_purpose(trip_data,'ptype_label','person type',3,1200)

C:\Users\Modeller\AppData\Local\Temp\ipykernel_40580\3385927089.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



- removed survey trips with no matching tours

In [8]:
trip_wTour = trip_data[trip_data['tour_id'].notna()].copy()

In [9]:
plot_purpose(trip_wTour,'tour_type','tour purpose',3,1200)

C:\Users\Modeller\AppData\Local\Temp\ipykernel_40580\3385927089.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\Modeller\AppData\Local\Temp\ipykernel_40580\3385927089.py:3: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



- removed survey trips with no matching tours

In [10]:
df_plot = trip_wTour.groupby(['source','tour_direction','purpose'])['trip_weight'].sum().reset_index()
df_plot['percentage'] = df_plot.groupby(['source','tour_direction'], group_keys=False)['trip_weight']. \
    apply(lambda x: x / float(x.sum()))

fig = px.bar(df_plot,
             x="percentage", y="purpose", color="source",barmode="group",
             facet_col='tour_direction', facet_col_wrap=2, orientation='h',
             category_orders={"tour_direction": ["outbound", "inbound"]},
             title="Trip purpose by " + 'tour_direction')
fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
fig.update_layout(height=500, width=770)
fig.for_each_xaxis(lambda a: a.update(tickformat = ".1%"))
fig.update_yaxes()
fig.show()

C:\Users\Modeller\AppData\Local\Temp\ipykernel_40580\707978159.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

